In [16]:
# Standard library
import os
import json
import time
import warnings
from typing import List, Dict, Any, Optional, Tuple
from collections import deque

# Third-party
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack core
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import ChatMessage

# Haystack components
from haystack.components.builders import PromptBuilder, ChatPromptBuilder
from haystack.components.generators.azure import AzureOpenAIGenerator
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever, InMemoryEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.joiners import DocumentJoiner

# Haystack routing and conditional logic
from haystack.components.routers import ConditionalRouter

# Web search
from haystack.components.websearch import SerperDevWebSearch
from haystack.components.converters import OutputAdapter

warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


In [4]:
def create_company_kb() -> List[Document]:
    """Create sample company knowledge base."""
    docs_data = [
        {"title": "Leave Policy", "content": "Employees get 20 days annual leave. Sick leave is 10 days per year."},
        {"title": "Remote Work", "content": "Remote work allowed up to 3 days/week with manager approval."},
        {"title": "Benefits", "content": "Health insurance covers employee and family. Gym membership included."},
        {"title": "Working Hours", "content": "Standard hours: 9 AM - 5 PM. Flexible hours available."},
        {"title": "Expenses", "content": "Business expenses reimbursed within 30 days. Keep all receipts."},
    ]
    return [Document(content=d["content"], meta={"title": d["title"]}) for d in docs_data]

company_docs = create_company_kb()
company_store = InMemoryDocumentStore()
company_store.write_documents(company_docs)

print(f"✅ Created knowledge base with {len(company_docs)} documents")

✅ Created knowledge base with 5 documents


In [13]:
routes = [
    {
        "condition": "{{ 'price' in (query | lower) or 'cost' in (query | lower) }}",
        "output": "{{ query }}",
        "output_name": "pricing",
        "output_type": str,
    },
    {
        "condition": "{{ 'hours' in (query | lower) or 'working hours' in (query | lower) }}",
        "output": "{{ query }}",
        "output_name": "hours",
        "output_type": str,
    },
    {
        "condition": "{{ True }}",
        "output": "{{ query }}",
        "output_name": "general",
        "output_type": str,
    },
]


In [14]:
router = ConditionalRouter(routes=routes)
test_queries = [
    "How many vacation days do I get?",
    "Can I work remotely?",
    "What are the working hours?"
]
print("\nTesting Query Router:\n")
for query in test_queries:
    result = router.run(query=query)
    route_taken = list(result.keys())[0]
    print(f"Query: {query}")
    print(f"  → Routed to: {route_taken}\n")


Testing Query Router:

Query: How many vacation days do I get?
  → Routed to: general

Query: Can I work remotely?
  → Routed to: general

Query: What are the working hours?
  → Routed to: hours



In [15]:
hr_template = """
You are an HR assistant. Answer the employee's HR-related question.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ query }}

Provide a clear, policy-based answer.
Answer:
"""

general_template = """
You are a helpful assistant. Answer the question based on the context.

Context:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

Question: {{ query }}

Answer:
"""

print("✅ Created specialized templates for different query types")

✅ Created specialized templates for different query types


In [17]:
from haystack.utils import Secret
def create_routing_rag(query: str)-> str:
    """
    Simple routing RAG that chooses template based on query type.
    """
    if any(word in query.lower() for word in ['leave', 'vacation', 'sick']):
        template = hr_template
        print(f"🔀 Routing to HR pipeline")
    else:
        template = general_template
        print(f"🔀 Routing to general pipeline")

    pipeline = Pipeline()
    pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=company_store))
    pipeline.add_component("prompt", PromptBuilder(template=template))
    pipeline.add_component(
        "llm", AzureOpenAIGenerator(
                    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
                    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
                    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
                    api_version="2024-12-01-preview",
                    generation_kwargs={"temperature": 0.3},
                )
    )
    pipeline.connect("retriever.documents", "prompt.documents")
    pipeline.connect("prompt", "llm")
    
    result = pipeline.run({
        "retriever": {"query": query, "top_k": 2},
        "prompt": {"query": query}
    })
    
    return result['llm']['replies'][0]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    answer = create_routing_rag(query)
    print(f"Answer: {answer}")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Query: How many vacation days do I get?
🔀 Routing to HR pipeline


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Answer: You are entitled to **20 days of annual leave** per year, as outlined in the company policy. Let me know if you need assistance with requesting or scheduling your vacation days!

Query: Can I work remotely?
🔀 Routing to general pipeline


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Answer: Yes, you can work remotely for up to 3 days per week, provided you have approval from your manager.

Query: What are the working hours?
🔀 Routing to general pipeline
Answer: The standard working hours are 9 AM to 5 PM, but flexible hours are available.


#### 3. Web Search Fallback

In [18]:
def should_use_web_search(query: str, retrieved_docs: List[Document], threshold: float = 0.3) -> bool:
    """
    Determine if web search is needed based on retrieval confidence.
    
    Args:
        query: User query
        retrieved_docs: Documents from local retrieval
        threshold: Minimum score for confidence
        
    Returns:
        True if web search should be used
    """
    if not retrieved_docs:
        return True
    if retrieved_docs[0].score < threshold:
        return True
    return False

retriever = InMemoryBM25Retriever(document_store=company_store)

test_cases = [
    "How many vacation days do I get?",  # Should find in KB
    "What's the weather like today?",  # Not in KB
    "Who won the latest Nobel Prize?",  # Not in KB
]

print("Web Search Decision Logic:\n")
for query in test_cases:
    result = retriever.run(query=query, top_k=3)
    docs = result['documents']
    use_web = should_use_web_search(query, docs)
    
    print(f"Query: {query}")
    if docs:
        print(f"  Top doc score: {docs[0].score:.3f}")
    print(f"  Use web search: {'✅ Yes' if use_web else '❌ No'}\n")

Web Search Decision Logic:

Query: How many vacation days do I get?
  Top doc score: 2.423
  Use web search: ❌ No

Query: What's the weather like today?
  Use web search: ✅ Yes

Query: Who won the latest Nobel Prize?
  Use web search: ✅ Yes



In [20]:
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
if SERPER_API_KEY:
    print(f"Serper Key: {SERPER_API_KEY[:20]}...")
else:
    print("⚠️  SerperDev API key not found (needed for web fallback example)")
    print("   Get free key at: https://serper.dev")

Serper Key: cc8a716899018b1b59d5...


In [21]:
def rag_with_web_fallback(query: str) -> Dict[str, Any]:
    """
    RAG pipeline with web search fallback.
    Falls back to LLM's knowledge if web search unavailable.
    """

    retriever = InMemoryBM25Retriever(document_store=company_store)
    retrieval_result = retriever.run(query=query, top_k=3)
    docs = retrieval_result['documents']

    use_web = should_use_web_search(query, docs, threshold=0.5)

    if use_web:
        print("🌐 Local knowledge insufficient - using web fallback")
        if SERPER_API_KEY:
            try:
                web_search = SerperDevWebSearch(api_key=Secret.from_env_var("SERPER_API_KEY"), top_k=3)
                web_result = web_search.run(query=query)

                web_doc = []
                for item in web_result.get('documents', []):
                    
                    web_doc.append(Document(
                        content=item.get('snippet', ''),
                        meta={'source': 'web', 'url': item.get('link', '')}
                    ))
                docs = web_doc
                source="web_search"
            except Exception as e:
                    print(f"  Web search failed: {e}")
                    print("  Falling back to LLM general knowledge")
                    docs = []
                    source = "llm_knowledge"
        else:
            print("  No web search API configured")
            print("  Using LLM's general knowledge")
            docs = []
            source = "llm_knowledge"

    else:
        print("📚 Using local knowledge base")
        source = "local_kb"

    if docs:
        template = """
        Answer the question based on the provided information.
        
        Information:
        {% for doc in documents %}
        {{ doc.content }}
        {% endfor %}
        
        Question: {{ query }}
        
        Answer:
        """
        prompt_builder = PromptBuilder(template=template)
        prompt = prompt_builder.run(documents=docs, query=query)['prompt']
    else:
        # No context, direct query
        prompt = f"Question: {query}\n\nAnswer:"

    llm = AzureOpenAIGenerator(
            api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version="2024-12-01-preview",
            generation_kwargs={"temperature": 0.3},
        )
    answer = llm.run(prompt=prompt)['replies'][0]

    return {
        'answer': answer,
        'source': source,
        'num_docs': len(docs),
        'docs': docs
    }

test_queries = [
    "How many vacation days do I get?",
    "What's the weather like today?",
    "Explain quantum computing"
]
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    result = rag_with_web_fallback(query)
    
    print(f"\nSource: {result['source']}")
    print(f"Documents used: {result['num_docs']}")
    print(f"\nAnswer:\n{result['answer']}")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Query: How many vacation days do I get?
📚 Using local knowledge base

Source: local_kb
Documents used: 3

Answer:
You get 20 vacation days (annual leave) per year.

Query: What's the weather like today?
🌐 Local knowledge insufficient - using web fallback
  Web search failed: 'Document' object has no attribute 'get'
  Falling back to LLM general knowledge

Source: llm_knowledge
Documents used: 0

Answer:
I'm sorry, I can't provide real-time weather updates. To find out the weather for today, you can check a reliable weather website or app like Weather.com, AccuWeather, or your local meteorological service.

Query: Explain quantum computing
🌐 Local knowledge insufficient - using web fallback
  Web search failed: 'Document' object has no attribute 'get'
  Falling back to LLM general knowledge

Source: llm_knowledge
Documents used: 0

Answer:
Quantum computing is a revolutionary approach to computation that leverages the principles of quantum mechanics, the fundamental theory in physics t

#### Self-Correcting RAG (Self-RAG)

In [22]:
def assess_document_relevance(query: str, doc_content: str) -> Tuple[bool, str]:
    """
    Use LLM to assess if document is relevant to query.
    
    Returns:
        (is_relevant: bool, reasoning: str)
    """
    prompt = f"""
    Is the following document relevant for answering the query?
    
    Query: {query}
    
    Document: {doc_content}
    
    Respond with ONLY 'YES' or 'NO', followed by a brief reason.
    Format: YES/NO - [reason]
    """

    llm = AzureOpenAIGenerator(
            api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version="2024-12-01-preview",
            generation_kwargs={"temperature": 0.3},
        )
    response = llm.run(prompt=prompt)['replies'][0]
    is_relevant = response.strip().upper().startswith('YES')

    return is_relevant, response

print("Testing Relevance Assessment:\n")

query = "How many vacation days do employees get?"
relevant_doc = "Employees get 20 days annual leave."
irrelevant_doc = "The office is located at 123 Main Street."

is_rel, reason = assess_document_relevance(query, relevant_doc)
print(f"Query: {query}")
print(f"Doc: {relevant_doc}")
print(f"Relevant: {is_rel}")
print(f"Reasoning: {reason}\n")

is_rel, reason = assess_document_relevance(query, irrelevant_doc)
print(f"Doc: {irrelevant_doc}")
print(f"Relevant: {is_rel}")
print(f"Reasoning: {reason}")

Testing Relevance Assessment:

Query: How many vacation days do employees get?
Doc: Employees get 20 days annual leave.
Relevant: True
Reasoning: YES - The document specifies the number of vacation days employees receive, which directly answers the query.

Doc: The office is located at 123 Main Street.
Relevant: False
Reasoning: NO - The document does not mention vacation days or any information related to employee benefits.


In [23]:
def validate_answer_grounding(answer: str, documents: List[Document]) -> Tuple[bool, str]:
    """
    Check if answer is grounded in provided documents.
    
    Returns:
        (is_grounded: bool, reasoning: str)
    """
    context = "\n".join([doc.content for doc in documents])
    
    prompt = f"""
    Is the following answer supported by the provided context?
    
    Context:
    {context}
    
    Answer:
    {answer}
    
    Respond with ONLY 'YES' if the answer is fully supported by the context, or 'NO' if it contains information not in the context or contradicts it.
    Format: YES/NO - [brief explanation]
    """
    llm = AzureOpenAIGenerator(
            api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version="2024-12-01-preview",
            generation_kwargs={"temperature": 0.3},
        )
    response = llm.run(prompt=prompt)['replies'][0]
    is_grounded = response.strip().upper().startswith('YES')
    
    return is_grounded, response

# Test answer validation
print("\nTesting Answer Validation:\n")

docs = [Document(content="Employees receive 20 days of annual vacation.")]
grounded_answer = "Employees get 20 vacation days per year."
hallucinated_answer = "Employees get 30 vacation days and unlimited sick leave."

is_valid, reason = validate_answer_grounding(grounded_answer, docs)
print(f"Answer: {grounded_answer}")
print(f"Grounded: {is_valid}")
print(f"Reasoning: {reason}\n")

is_valid, reason = validate_answer_grounding(hallucinated_answer, docs)
print(f"Answer: {hallucinated_answer}")
print(f"Grounded: {is_valid}")
print(f"Reasoning: {reason}")


Testing Answer Validation:

Answer: Employees get 20 vacation days per year.
Grounded: True
Reasoning: YES - The answer is fully supported by the context, as both state that employees receive 20 days of annual vacation.

Answer: Employees get 30 vacation days and unlimited sick leave.
Grounded: False
Reasoning: NO - The answer contradicts the context by stating employees get 30 vacation days, while the context specifies 20 days of annual vacation. It also introduces "unlimited sick leave," which is not mentioned in the context.


#### Self-Correcting RAG (Self-RAG)

In [24]:
def assess_document_relevance(query: str, doc_content: str) -> Tuple[bool, str]:
    """
    Use LLM to assess if document is relevant to query.
    
    Returns:
        (is_relevant: bool, reasoning: str)
    """

    prompt = f"""
    Is the following document relevant for answering the query?
    
    Query: {query}
    
    Document: {doc_content}
    
    Respond with ONLY 'YES' or 'NO', followed by a brief reason.
    Format: YES/NO - [reason]
    """

    llm = AzureOpenAIGenerator(
            api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version="2024-12-01-preview",
            generation_kwargs={"temperature": 0.3},
        )
    
    response = llm.run(prompt=prompt)['replies'][0]
    
    is_relevant = response.strip().upper().startswith('YES')
    
    return is_relevant, response

# Test relevance assessment
print("Testing Relevance Assessment:\n")

query = "How many vacation days do employees get?"
relevant_doc = "Employees get 20 days annual leave."
irrelevant_doc = "The office is located at 123 Main Street."

is_rel, reason = assess_document_relevance(query, relevant_doc)
print(f"Query: {query}")
print(f"Doc: {relevant_doc}")
print(f"Relevant: {is_rel}")
print(f"Reasoning: {reason}\n")

is_rel, reason = assess_document_relevance(query, irrelevant_doc)
print(f"Doc: {irrelevant_doc}")
print(f"Relevant: {is_rel}")
print(f"Reasoning: {reason}")

Testing Relevance Assessment:

Query: How many vacation days do employees get?
Doc: Employees get 20 days annual leave.
Relevant: True
Reasoning: YES - The document directly states the number of vacation days employees receive, which answers the query.

Doc: The office is located at 123 Main Street.
Relevant: False
Reasoning: NO - [The document does not contain information about vacation days for employees.]


In [25]:
def validate_answer_grounding(answer: str, documents: List[Document]) -> Tuple[bool, str]:
    """
    Check if answer is grounded in provided documents.
    
    Returns:
        (is_grounded: bool, reasoning: str)
    """
    context = "\n".join([doc.content for doc in documents])
    
    prompt = f"""
    Is the following answer supported by the provided context?
    
    Context:
    {context}
    
    Answer:
    {answer}
    
    Respond with ONLY 'YES' if the answer is fully supported by the context, or 'NO' if it contains information not in the context or contradicts it.
    Format: YES/NO - [brief explanation]
    """
    llm = AzureOpenAIGenerator(
            api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version="2024-12-01-preview",
            generation_kwargs={"temperature": 0.3},
        )
    
    response = llm.run(prompt=prompt)['replies'][0]
    is_grounded = response.strip().upper().startswith('YES')
    
    return is_grounded, response

# Test answer validation
print("\nTesting Answer Validation:\n")

docs = [Document(content="Employees receive 20 days of annual vacation.")]
grounded_answer = "Employees get 20 vacation days per year."
hallucinated_answer = "Employees get 30 vacation days and unlimited sick leave."

is_valid, reason = validate_answer_grounding(grounded_answer, docs)
print(f"Answer: {grounded_answer}")
print(f"Grounded: {is_valid}")
print(f"Reasoning: {reason}\n")

is_valid, reason = validate_answer_grounding(hallucinated_answer, docs)
print(f"Answer: {hallucinated_answer}")
print(f"Grounded: {is_valid}")
print(f"Reasoning: {reason}")
    



Testing Answer Validation:

Answer: Employees get 20 vacation days per year.
Grounded: True
Reasoning: YES - The answer is fully supported by the context, as both state that employees receive 20 days of annual vacation.

Answer: Employees get 30 vacation days and unlimited sick leave.
Grounded: False
Reasoning: NO - The context states employees receive 20 days of annual vacation, while the answer claims they get 30 vacation days and unlimited sick leave, which is not supported by the context.
